In [2]:
pip install tree-of-thoughts-llm

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 2.8 MB/s eta 0:00:00a 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 4.7 MB/s eta 0:00:0000:010:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... error
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [33 lines of output]
      Traceback (most recent call last):
        File "/opt/anaconda3/envs/torch-gpu/lib/python3.12/s

In [1]:
from typing import List, Tuple
import copy

# -------------------------- ToT核心模块 --------------------------
class ToTNode:
    """思维树节点：存储当前数独状态、父节点、当前步骤"""
    def __init__(self, board: List[List[int]], parent: 'ToTNode' = None, step: int = 0):
        self.board = board          # 当前数独盘面
        self.parent = parent        # 父节点（回溯用）
        self.step = step            # 当前步数
        self.children = []          # 子节点（候选分支）

    def is_terminal(self) -> bool:
        """判断是否为终态（数独已填满）"""
        return all(cell != 0 for row in self.board for cell in row)

    def get_empty_cell(self) -> Tuple[int, int]:
        """找到第一个空单元格（0的位置）"""
        for i in range(9):
            for j in range(9):
                if self.board[i][j] == 0:
                    return (i, j)
        return (-1, -1)

    def is_valid(self, row: int, col: int, num: int) -> bool:
        """评估：判断在(row,col)填num是否符合数独规则（剪枝核心）"""
        # 检查行
        if num in self.board[row]:
            return False
        # 检查列
        if num in [self.board[i][col] for i in range(9)]:
            return False
        # 检查3x3宫格
        start_row, start_col = 3 * (row // 3), 3 * (col // 3)
        for i in range(start_row, start_row + 3):
            for j in range(start_col, start_col + 3):
                if self.board[i][j] == num:
                    return False
        return True

    def generate_children(self) -> List['ToTNode']:
        """生成子节点：所有合法的候选填数分支（ToT生成环节）"""
        if self.is_terminal():
            return []
        row, col = self.get_empty_cell()
        children = []
        # 生成1-9的候选数，只保留合法的（剪枝）
        for num in range(1, 10):
            if self.is_valid(row, col, num):
                new_board = copy.deepcopy(self.board)
                new_board[row][col] = num
                child_node = ToTNode(new_board, parent=self, step=self.step + 1)
                children.append(child_node)
        self.children = children
        return children

class ToTSolver:
    """ToT求解器：用深度优先搜索(DFS)遍历思维树"""
    def __init__(self, root_board: List[List[int]]):
        self.root = ToTNode(root_board)  # 根节点：初始数独盘面

    def solve(self) -> List[List[int]]:
        """DFS遍历思维树，找到终态解"""
        stack = [self.root]
        while stack:
            current_node = stack.pop()
            # 找到终态，回溯得到完整解
            if current_node.is_terminal():
                return current_node.board
            # 生成子节点，入栈（DFS：后进先出）
            children = current_node.generate_children()
            stack.extend(children)
        return []  # 无解

# -------------------------- 工具函数 --------------------------
def print_board(board: List[List[int]]):
    """格式化打印数独盘面"""
    for i in range(9):
        if i % 3 == 0 and i != 0:
            print("- - - - - - - - - - - - ")
        row = ""
        for j in range(9):
            if j % 3 == 0 and j != 0:
                row += "| "
            row += str(board[i][j]) + " "
        print(row)

# -------------------------- 主程序：测试用例 --------------------------
if __name__ == "__main__":
    # 示例数独题（0代表空单元格）
    example_sudoku = [
        [5, 3, 0, 0, 7, 0, 0, 0, 0],
        [6, 0, 0, 1, 9, 5, 0, 0, 0],
        [0, 9, 8, 0, 0, 0, 0, 6, 0],
        [8, 0, 0, 0, 6, 0, 0, 0, 3],
        [4, 0, 0, 8, 0, 3, 0, 0, 1],
        [7, 0, 0, 0, 2, 0, 0, 0, 6],
        [0, 6, 0, 0, 0, 0, 2, 8, 0],
        [0, 0, 0, 4, 1, 9, 0, 0, 5],
        [0, 0, 0, 0, 8, 0, 0, 7, 9]
    ]

    print("初始数独盘面：")
    print_board(example_sudoku)

    # 用ToT求解
    solver = ToTSolver(example_sudoku)
    solution = solver.solve()

    if solution:
        print("\nToT求解结果：")
        print_board(solution)
    else:
        print("\n无解")

初始数独盘面：
5 3 0 | 0 7 0 | 0 0 0 
6 0 0 | 1 9 5 | 0 0 0 
0 9 8 | 0 0 0 | 0 6 0 
- - - - - - - - - - - - 
8 0 0 | 0 6 0 | 0 0 3 
4 0 0 | 8 0 3 | 0 0 1 
7 0 0 | 0 2 0 | 0 0 6 
- - - - - - - - - - - - 
0 6 0 | 0 0 0 | 2 8 0 
0 0 0 | 4 1 9 | 0 0 5 
0 0 0 | 0 8 0 | 0 7 9 

ToT求解结果：
5 3 4 | 6 7 8 | 9 1 2 
6 7 2 | 1 9 5 | 3 4 8 
1 9 8 | 3 4 2 | 5 6 7 
- - - - - - - - - - - - 
8 5 9 | 7 6 1 | 4 2 3 
4 2 6 | 8 5 3 | 7 9 1 
7 1 3 | 9 2 4 | 8 5 6 
- - - - - - - - - - - - 
9 6 1 | 5 3 7 | 2 8 4 
2 8 7 | 4 1 9 | 6 3 5 
3 4 5 | 2 8 6 | 1 7 9 
